## TOGSim Configuration
The machine is one TOGSim machine file, named by `TORCHSIM_TARGET`: the compiler compiles for it
and TOGSim times on it. Shipped machines are in `$TORCHSIM_COMPILE_DIR/third_party/TOGSim/configs`.

The workload is a batched GeMM, 8 x 1024 x 1024 x 1024 in bf16. It is compute bound: each tile does
many multiply-accumulates per byte it loads, so one core keeps its systolic arrays busy, and on two
cores the batches split between them. (A 4096 x 4096 mm, by contrast, is bound by DRAM bandwidth
once two cores share it.) Each cell takes about two minutes.

In [ ]:
import os
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_timing_only.yml")
os.environ["TORCHSIM_LOG_PATH"] = os.path.join(os.getcwd(), "togsim_results")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "outputs")
import glob
import torch

def latest_log():
    """The newest TOGSim log under TORCHSIM_LOG_PATH."""
    return max(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")), key=os.path.getmtime)

_seen = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")))

def new_logs():
    """The TOGSim logs written since the last call, oldest first: one per simulation, two for a
    resident mm (its operand's copy-in, then its body), whose cycles add up."""
    logs = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log"))) - _seen
    _seen.update(logs)
    return sorted(logs, key=os.path.getmtime)

### Single Core

In [ ]:
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "single_core")
device = torch.device("npu:0")

input = torch.randn(8, 1024, 1024, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(8, 1024, 1024, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.bmm)
npu_outut = opt_fn(input, weight)

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}

### Multi-Core

In [ ]:
!diff togsim_configs/togsim_config_timing_only.yml togsim_configs/togsim_config_2_cores.yml

In [ ]:
torch._dynamo.reset()
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_2_cores.yml")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "multi_core")

input = torch.randn(8, 1024, 1024, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(8, 1024, 1024, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.bmm)
npu_outut = opt_fn(input, weight)

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}